In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Verify GPU availability

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("GPU not detected. Check Kaggle notebook settings.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU count: 2


data set verification

In [2]:
import os

DATASET_PATH = "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10"

print("Dataset exists:", os.path.isdir(DATASET_PATH))

for item in [
    "data.yaml",
    "train/images",
    "train/labels",
    "valid/images",
    "valid/labels",
    "test/images",
    "test/labels"
]:
    path = os.path.join(DATASET_PATH, item)
    print(f"{'✅ Found' if os.path.exists(path) else '❌ Missing'}: {item}")

Dataset exists: True
✅ Found: data.yaml
✅ Found: train/images
✅ Found: train/labels
✅ Found: valid/images
✅ Found: valid/labels
✅ Found: test/images
✅ Found: test/labels


Install Ultralytics****

In [2]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


In [3]:
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.192


In [3]:
import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Check Kaggle Settings > Accelerator.")

PyTorch: 2.11.0+cu128
Ultralytics: 8.4.191
CUDA available: True
GPU: Tesla T4


In [4]:
import os

DATASET_PATH = "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10"

print("Dataset path exists:", os.path.exists(DATASET_PATH))

if os.path.exists(DATASET_PATH):
    print("\nDataset contents:")
    print(os.listdir(DATASET_PATH))
else:
    print("\nDataset path not found.")
    print("\nAvailable input folders:")
    print(os.listdir("/kaggle/input"))



Dataset path exists: True

Dataset contents:
['class_counts_0_to_10.csv', 'class_distribution.csv', 'data.yaml', 'test', 'train', 'valid']


In [5]:
import os

DATASET_PATH = "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10"

for split in ["train", "valid", "test"]:
    print(f"\n--- {split.upper()} ---")

    for folder in ["images", "labels"]:
        path = os.path.join(DATASET_PATH, split, folder)

        if os.path.exists(path):
            files = os.listdir(path)
            print(f"{folder}: {len(files)} files")
        else:
            print(f"WARNING: {path} not found")


--- TRAIN ---
images: 2148 files
labels: 2148 files

--- VALID ---
images: 614 files
labels: 614 files

--- TEST ---
images: 307 files
labels: 307 files


In [6]:
import yaml
import os

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

print(yaml.dump(data, sort_keys=False))

print("\nNumber of classes:", data.get("nc"))
print("Class names:", data.get("names"))

path: /content/drive/MyDrive/CNN/merged_vehicle_dataset_11class
train: train/images
val: valid/images
test: test/images
names:
  0: bicycle
  1: bus
  2: car
  3: bike
  4: auto
  5: truck
  6: van
  7: person
  8: concrete_mixer
  9: tractor
  10: vehicle_truck
nc: 11


Number of classes: 11
Class names: {0: 'bicycle', 1: 'bus', 2: 'car', 3: 'bike', 4: 'auto', 5: 'truck', 6: 'van', 7: 'person', 8: 'concrete_mixer', 9: 'tractor', 10: 'vehicle_truck'}


In [7]:
import os
import yaml

# Actual dataset path in Kaggle
DATASET_PATH = "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10"

# Check that the required folders exist
for split in ["train", "valid", "test"]:
    image_path = os.path.join(DATASET_PATH, split, "images")
    label_path = os.path.join(DATASET_PATH, split, "labels")

    assert os.path.isdir(image_path), f"Missing folder: {image_path}"
    assert os.path.isdir(label_path), f"Missing folder: {label_path}"

print("Dataset folders verified!")

# Define the 11 classes
names = [
    "bicycle",
    "bus",
    "car",
    "bike",
    "auto",
    "truck",
    "van",
    "person",
    "concrete_mixer",
    "tractor",
    "vehicle_truck"
]

# Create a Kaggle-compatible configuration
config = {
    "path": DATASET_PATH,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(names),
    "names": names
}

# Save to Kaggle's writable working directory
DATA_YAML = "/kaggle/working/vehicle_data.yaml"

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("\nNew configuration:")
print(open(DATA_YAML).read())

Dataset folders verified!

New configuration:
path: /kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10
train: train/images
val: valid/images
test: test/images
nc: 11
names:
- bicycle
- bus
- car
- bike
- auto
- truck
- van
- person
- concrete_mixer
- tractor
- vehicle_truck



TRAINING

In [8]:
import torch
from ultralytics import YOLO

# Check GPU
assert torch.cuda.is_available(), "GPU not available!"

print("GPU:", torch.cuda.get_device_name(0))
print("Starting YOLO11n training...")

# Load pretrained YOLO11n model
model = YOLO("yolo11n.pt")

# Train the model
results = model.train(
    data="/kaggle/working/vehicle_data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=10,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_detection",
    seed=42,
    plots=True,
    save=True
)

print("Training completed!")

GPU: Tesla T4
Starting YOLO11n training...
Ultralytics 8.4.191 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/vehicle_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_dete

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      2.85G       1.42      1.681      1.374         49        640: 100% ━━━━━━━━━━━━ 134/134 5.8it/s 23.2s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 3.1it/s 6.5s0.2s
                   all        611       2022      0.326       0.32      0.254      0.149

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.85G      1.432      1.612      1.381         48        640: 100% ━━━━━━━━━━━━ 134/134 5.8it/s 23.3s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 3.1it/s 6.5s0.3s
                   all        611       2022      0.364      0.306      0.263      0.153

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      2.85G       1.38      1.534      1.347         54        640: 100% ━━━━━━━━━━━━ 134/134 5.8it/s 23.0s0.1s
                 Class     Images

adding 50 more ephocs

In [9]:
from ultralytics import YOLO
import os
import yaml

# 1. Dataset configuration
dataset_path = "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10"
yaml_path = "/kaggle/working/vehicle_data.yaml"

data = {
    "path": dataset_path,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 11,
    "names": [
        "bicycle", "bus", "car", "bike", "auto", "truck",
        "van", "person", "concrete_mixer", "tractor", "vehicle_truck"
    ]
}

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

# 2. Load your existing best YOLO11n model
best_model_path = "/kaggle/working/yolo11n_runs/vehicle_detection/weights/best.pt"

assert os.path.exists(best_model_path), (
    f"Model not found: {best_model_path}. "
    "Make sure your previous training output still exists."
)

model = YOLO(best_model_path)

# 3. Fine-tune for 50 additional epochs
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,       # Tesla T4 GPU
    workers=2,
    patience=10,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_detection_finetune",
    seed=42,
    plots=True,
    save=True
)

# 4. Validate the best checkpoint
best_path = "/kaggle/working/yolo11n_runs/vehicle_detection_finetune/weights/best.pt"

best_model = YOLO(best_path)
metrics = best_model.val(data=yaml_path, split="val", imgsz=640, batch=16, device=0)

print("\nFine-tuning complete!")
print("Best model:", best_path)
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")

AssertionError: Model not found: /kaggle/working/yolo11n_runs/vehicle_detection/weights/best.pt. Make sure your previous training output still exists.

| Setting | Value | Purpose |
|---|---:|---|
| Model | YOLO11n | Lightweight object detection |
| Epochs | 50 | Maximum number of complete dataset passes |
| Image size | 640 | Input image resolution |
| Batch size | 16 | Images processed per training batch |
| GPU | Tesla T4 | Accelerates training |
| Patience | 10 | Stops early if validation fitness doesn't improve |
| Seed | 42 | Helps make experiments reproducible |


In [9]:
import torch
from ultralytics import YOLO

# Check GPU
assert torch.cuda.is_available(), "GPU not available!"

print("GPU:", torch.cuda.get_device_name(0))
print("Starting YOLO11n training...")

# Load pretrained YOLO11n model
model = YOLO("yolo11n.pt")

# Train the model
results = model.train(
    data="/kaggle/working/vehicle_data.yaml",
    epochs=100,
    imgsz=800,
    batch=8,
    device=0,
    workers=2,
    patience=10,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_detection",
    seed=42,
    plots=True,
    save=True
)

print("Training completed!")

GPU: Tesla T4
Starting YOLO11n training...
Ultralytics 8.4.191 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/vehicle_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_dete

# import torch
from ultralytics import YOLO

# Check GPU
assert torch.cuda.is_available(), "GPU not available!"

print("GPU:", torch.cuda.get_device_name(0))
print("Starting YOLO11n training...")

# Load pretrained YOLO11n model
model = YOLO("yolo11n.pt")

# Train the model
results = model.train(
    data="/kaggle/working/vehicle_data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=10,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_detection",
    seed=42,
    plots=True,
    save=True
)

print("Training completed!")

In [5]:
from pathlib import Path
import yaml

dataset_path = "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10"

data = {
    "path": dataset_path,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 11,
    "names": [
        "bicycle", "bus", "car", "bike", "auto",
        "truck", "van", "person", "concrete_mixer",
        "tractor", "vehicle_truck"
    ]
}

yaml_path = "/kaggle/working/vehicle_data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

# Verify the YAML and dataset paths
assert Path(dataset_path).exists(), f"Dataset not found: {dataset_path}"
assert Path(yaml_path).exists(), "YAML creation failed"
assert (Path(dataset_path) / "train/images").exists(), "Training images folder missing"
assert (Path(dataset_path) / "valid/images").exists(), "Validation images folder missing"

print("YAML created successfully:", yaml_path)
print("Dataset paths verified!")

YAML created successfully: /kaggle/working/vehicle_data.yaml
Dataset paths verified!


In [6]:
import os
import torch
from ultralytics import YOLO

# Check GPU
assert torch.cuda.is_available(), "GPU not available!"
print("GPU:", torch.cuda.get_device_name(0))

# Load model
model = YOLO("yolo11s.pt")

# Train
results = model.train(
    data="/kaggle/working/vehicle_data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    cos_lr=True,
    close_mosaic=10,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_yolo11s_experiment",
    seed=42,
    plots=True,
    save=True,
    save_period=5
)

# Checkpoint paths
run_dir = "/kaggle/working/yolo11n_runs/vehicle_yolo11s_experiment"
best_path = os.path.join(run_dir, "weights", "best.pt")
last_path = os.path.join(run_dir, "weights", "last.pt")

# Copy best model to an easy-to-find location
if os.path.exists(best_path):
    import shutil
    shutil.copy2(best_path, "/kaggle/working/best_vehicle_yolo11s.pt")
    print("Best model saved to: /kaggle/working/best_vehicle_yolo11s.pt")

if os.path.exists(last_path):
    print("Last checkpoint saved to:", last_path)

print("Training finished.")

GPU: Tesla T4
Ultralytics 8.4.192 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/vehicle_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_yolo11s_experiment-2, nbs=64, nms

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      2.67G      1.555      1.913      1.529         31        640: 83% ━━━━━━━━━━── 224/268 9.5it/s 24.7s<4.6ss


KeyboardInterrupt: 

| Setting | Meaning | Why it matters |
|---|---|---|
| `epochs=100` | Train for up to 100 complete passes through your dataset. | More training can improve accuracy, but not always. |
| `imgsz=640` | Resize images to 640 × 640 pixels for training. | Larger images may help detect smaller vehicles but use more GPU memory. |
| `batch=8` | Process 8 images per training batch. | A smaller batch uses less GPU memory. |
| `device=0` | Train on GPU 0. | Uses your Kaggle GPU instead of CPU. |
| `workers=2` | Use 2 workers to load training images. | Helps prepare data while the GPU trains. |
| `patience=20` | Stop early if validation fitness doesn't improve for 20 consecutive epochs. | Avoids wasting time when training has stopped improving. |
| `optimizer="AdamW"` | Use the AdamW optimization algorithm. | Controls how model weights are updated during training. |
| `lr0=0.001` | Set the initial learning rate to 0.001. | Controls the size of weight updates. Too high or too low can hurt learning. |
| `weight_decay=0.0005` | Apply weight decay of 0.0005. | Helps reduce overfitting. |
| `cos_lr=True` | Use a cosine learning-rate schedule. | Gradually decreases the learning rate during training. |
| `close_mosaic=10` | Disable mosaic augmentation for the final 10 epochs. | Lets the model finish training on images closer to normal conditions. |
| `project="/kaggle/working/yolo11n_runs"` | Choose the main output folder. | Stores training results and checkpoints here. |
| `name="vehicle_yolo11s_experiment"` | Name this training run. | Keeps its files separate from other experiments. |
| `seed=42` | Set a random seed. | Helps make experiments more reproducible. |
| `plots=True` | Generate training graphs. | Helps you inspect loss and validation metrics. |
| `save=True` | Save model checkpoints. | Preserves trained model weights. |
| `save_period=5` | Save a checkpoint every 5 epochs. | Provides periodic checkpoints during training. |


In [7]:
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("GPU available:", torch.cuda.is_available())

Ultralytics: 8.4.192
GPU available: True


In [8]:
from ultralytics import YOLO
import torch
from pathlib import Path

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings first."
assert Path("/kaggle/working/vehicle_data.yaml").exists(), \
    "Recreate vehicle_data.yaml before training."

model = YOLO("yolo26n.pt")

results = model.train(
    data="/kaggle/working/vehicle_data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=10,
    project="/kaggle/working/yolo26_runs",
    name="vehicle_yolo26n",
    seed=42,
    plots=True,
    save=True
)

Ultralytics 8.4.192 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/vehicle_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_yolo26n, nbs=64, nms=None, opset=None, optimize

11n with 100 epochs

In [10]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/kaggle/working/vehicle_data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=15,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_detection_100epochs",
    seed=42,
    plots=True,
    save=True
)

Ultralytics 8.4.192 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/vehicle_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_detection_100epochs, nbs=64, nms=None, opset=N

In [11]:
from ultralytics import YOLO
import yaml

# Dataset configuration
data = {
    "path": "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 11,
    "names": [
        "bicycle", "bus", "car", "bike", "auto", "truck",
        "van", "person", "concrete_mixer", "tractor", "vehicle_truck"
    ]
}

yaml_path = "/kaggle/working/vehicle_data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

# Load the best trained model
model = YOLO(
    "/kaggle/working/yolo11n_runs/vehicle_detection_100epochs/weights/best.pt"
)

# Validate on validation dataset
results = model.val(
    data=yaml_path,
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_detection_validation"
)

# Print metrics
print("\n--- Validation Results ---")
print(f"Precision:   {results.box.mp:.4f}")
print(f"Recall:      {results.box.mr:.4f}")
print(f"mAP@50:      {results.box.map50:.4f}")
print(f"mAP@50-95:   {results.box.map:.4f}")

print("\nResults saved to:")
print("/kaggle/working/yolo11n_runs/vehicle_detection_validation")

Ultralytics 8.4.192 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,599,481 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 119.7±39.1 MB/s, size: 71.0 KB)
val: Scanning /kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10/valid/labels... 611 images, 118 backgrounds, 3 corrupt: 100% ━━━━━━━━━━━━ 614/614 460.0it/s 1.3s0.1s
val: /kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10/valid/images/indian_20210506_14_10_30_000_csn5ZBbKQsPS82BdtEmUny6qhf52_F_3000_4000.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10/valid/images/.indian_20210506_14_10_30_000_csn5ZBbKQsPS82BdtEmUny6qhf52_F_3000_4000.9c7c10d544ef4a8fb0d9e3a6e53ac42a.jpg'
val: /kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_

In [13]:
import yaml
import os

dataset_path = "/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10"

data = {
    "path": dataset_path,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 11,
    "names": [
        "bicycle", "bus", "car", "bike", "auto",
        "truck", "van", "person", "concrete_mixer",
        "tractor", "vehicle_truck"
    ]
}

yaml_path = "/kaggle/working/vehicle_dataset.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("YAML created:", yaml_path)
print("Dataset exists:", os.path.exists(dataset_path))
print("Test images exist:", os.path.exists(os.path.join(dataset_path, "test/images")))

YAML created: /kaggle/working/vehicle_dataset.yaml
Dataset exists: True
Test images exist: True


In [14]:
from ultralytics import YOLO

# Load your best trained model
model = YOLO(
    "/kaggle/working/yolo11n_runs/vehicle_detection_100epochs/weights/best.pt"
)

# Evaluate on the test dataset
results = model.val(
    data="/kaggle/working/vehicle_dataset.yaml",  # Update if your YAML is elsewhere
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/kaggle/working/yolo11n_runs",
    name="vehicle_detection_test"
)

print("\n--- Test Results ---")
print(f"Precision:   {results.box.mp:.4f}")
print(f"Recall:      {results.box.mr:.4f}")
print(f"mAP@50:      {results.box.map50:.4f}")
print(f"mAP@50-95:   {results.box.map:.4f}")

Ultralytics 8.4.192 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,599,481 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 0.3±0.1 MB/s, size: 53.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10/test/labels... 305 images, 72 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 307/307 7.8it/s 39.4s0.1sss
val: /kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10/test/images/xml_indian_20210514_12_09_56_000_5XOkot3crTZw22PEJp2NMSjGIc83_T_3456_4608.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/pyk999/vechicle/dataset/merged_vehicle_dataset_stratified_70_20_10/test/images/.xml_indian_20210514_12_09_56_000_5XOkot3c

In [15]:
import os

model_path = "/kaggle/working/yolo11n_runs/vehicle_detection_100epochs/weights/best.pt"

if os.path.exists(model_path):
    print("✅ Model found!")
    print(f"Model size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("❌ Model not found. Check your model path.")

✅ Model found!
Model size: 5.23 MB


In [16]:
import shutil

source = "/kaggle/working/yolo11n_runs/vehicle_detection_100epochs/weights/best.pt"
destination = "/kaggle/working/best_vehicle_yolo11n.pt"

shutil.copy2(source, destination)

print("✅ Model saved to:", destination)

✅ Model saved to: /kaggle/working/best_vehicle_yolo11n.pt


In [17]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/best_vehicle_yolo11n.pt"))

/kaggle/working/best_vehicle_yolo11n.pt

In [18]:
import shutil
import os

source = "/kaggle/working/yolo11n_runs/vehicle_detection_100epochs/weights/best.pt"
destination = "/kaggle/working/best_vehicle_yolo11n.pt"

if os.path.exists(source):
    shutil.copy2(source, destination)
    print("Model copied successfully!")
else:
    print("Original model not found. Check the path.")

Model copied successfully!


In [19]:
from IPython.display import FileLink, display

display(FileLink('/kaggle/working/best_vehicle_yolo11n.pt'))

/kaggle/working/best_vehicle_yolo11n.pt

In [20]:
import os
import zipfile

model_path = "/kaggle/working/best_vehicle_yolo11n.pt"
zip_path = "/kaggle/working/best_vehicle_yolo11n.zip"

if os.path.exists(model_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(model_path, arcname="best_vehicle_yolo11n.pt")

    print("ZIP created successfully!")
    print("Location:", zip_path)
    print("Size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")
else:
    print("Model file not found!")

ZIP created successfully!
Location: /kaggle/working/best_vehicle_yolo11n.zip
Size: 4.72 MB


test video data

In [ ]:
from ultralytics import YOLO
import os
import glob

# Load your trained model
model_path = "/kaggle/working/best_vehicle_yolo11n.pt"

if not os.path.exists(model_path):
    model_path = "/kaggle/working/yolo11n_runs/vehicle_detection_100epochs/weights/best.pt"

model = YOLO(model_path)
print("Model loaded successfully!")

# Folder containing your seven videos
video_dir = "/kaggle/input/datasets/pyk999/test-data-for-traffic"

videos = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))

print("Videos found:", len(videos))

# Output folder
output_dir = "/kaggle/working/traffic_video_results"
os.makedirs(output_dir, exist_ok=True)

# Process each video
for i, video_path in enumerate(videos, start=1):
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    print(f"\nProcessing video {i}/{len(videos)}: {video_name}")

    model.predict(
        source=video_path,
        imgsz=640,
        conf=0.25,
        save=True,
        project=output_dir,
        name=video_name,
        exist_ok=True,
        verbose=False
    )

    print("Completed:", video_name)

print("\nAll videos processed!")
print("Results folder:", output_dir)

Model loaded successfully!
Videos found: 7

Processing video 1/7: video_20261022_160935
WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to /kaggle/working/traffic_video_results/video_20261022_160935
Completed: video_20261022_160935

Processing video 2/7: video_20261022_161033
WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/pr